In [1]:
# === ENVIRONMENT & FILEPATH SETUP ===
import os
import sys
import ctypes

try:
    ctypes.CDLL("/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/cu13/lib/libnvJitLink.so.13")
except Exception:
    pass

codebase_path = "../"
if codebase_path not in sys.path:
    sys.path.insert(0, codebase_path)

for key in list(sys.modules.keys()):
    if key.startswith("src"):
        del sys.modules[key]

DATA_DIR = f"{codebase_path}/output/cache"
ENV_PATH = f"{codebase_path}/artifacts/.env"
MODELS_DIR = f"{codebase_path}/output/models"
ARTIFACTS_DIR = f"{codebase_path}/artifacts"
print("💻 Local Lab Server Environment Loaded.")

💻 Local Lab Server Environment Loaded.


## 1. Load Data & Base Model

In [2]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from src.inference.evaluate import run_evaluation

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

val_full = load_jsonl(f"{DATA_DIR}/val_full_info.jsonl")
val_struct = load_jsonl(f"{DATA_DIR}/val_structural.jsonl")
print(f"Loaded {len(val_full)} full-info validation samples and {len(val_struct)} structural validation samples.")


Loaded 2039 full-info validation samples and 2039 structural validation samples.


In [3]:
# === CONFIGURATION ===
COMPUTE_DTYPE = torch.bfloat16

MODEL_ID = "ibm-granite/granite-4.1-8b"

SFT_STRUCT_DIR = f"{MODELS_DIR}/granite-4.1-8b_structOnly_DoRA"
SFT_FULL_DIR   = f"{MODELS_DIR}/granite-4.1-8b_fullInfo_DoRA"

In [4]:
# === LOAD BASE MODEL ===
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True
)

if MODEL_ID == "mistralai/Mistral-Nemo-Instruct-2407":
	tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, 
											trust_remote_code=True,
											fix_mistral_regex=True
											)
else:
	tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, 
											trust_remote_code=True,
											)

tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    dtype=COMPUTE_DTYPE,
)
base_model.eval()
print("Base model loaded successfully!")


Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Base model loaded successfully!


## 2. Evaluate SFT Full-Information Model

In [5]:
# Load the Full-Info LoRA adapter
try:
    print(f"Loading adapter from {SFT_FULL_DIR}...")
    model_full = PeftModel.from_pretrained(base_model, SFT_FULL_DIR)
    
    # Evaluate on Full Info Dataset
    acc_sft_full, results_sft_full = run_evaluation(
        model=model_full,
        tokenizer=tokenizer,
        dataset=val_full,
        training_strategy="DoRA",
        prompt_format="fullInfo",
        model_name=MODEL_ID,
        output_csv=f"{ARTIFACTS_DIR}/experiment_summary.csv"
    )
    
    # Unload adapter to free memory for the next evaluation
    model_full.unload()
except Exception as e:
    print(f"Could not load or evaluate Full-Info SFT model: {e}")


Loading adapter from ..//output/models/granite-4.1-8b_fullInfo_DoRA...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Evaluating DoRA - fullInfo:   0%|          | 0/2039 [00:00<?, ?it/s]

/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Evaluating DoRA - fullInfo:   1%|          | 20/2039 [02:41<4:32:08,  8.09s/it]

...

Evaluating DoRA - fullInfo:  97%|█████████▋| 1980/2039 [3:25:37<04:13,  4.30s/it]

Evaluating DoRA - fullInfo:  98%|█████████▊| 2000/2039 [3:27:07<02:49,  4.36s/it]

Evaluating DoRA - fullInfo:  99%|█████████▉| 2020/2039 [3:28:29<01:21,  4.29s/it]

Evaluating DoRA - fullInfo: 100%|██████████| 2039/2039 [3:29:55<00:00,  6.18s/it]

\n--- Evaluation Results ---
Training Strategy: DoRA
Prompt Format: fullInfo
Model: ibm-granite/granite-4.1-8b
Accuracy: 0.1412
Format Error Rate: 0.9985
Semantic Confusion: 0.2165
Option Bias (A): 1.0000
Latency: 12595.11 seconds
Detailed predictions saved to: /data220_2/emmy/mlbio/hw4/output/validation/validation_granite-4.1-8b_fullInfo_DoRA.csv


## 3. Evaluate SFT Structural-Only Model

In [6]:
# Load the Structural-Only LoRA adapter
try:
    print(f"Loading adapter from {SFT_STRUCT_DIR}...")
    model_struct = PeftModel.from_pretrained(base_model, SFT_STRUCT_DIR)
    
    # Evaluate on Structural Only Dataset
    acc_sft_struct, results_sft_struct = run_evaluation(
        model=model_struct,
        tokenizer=tokenizer,
        dataset=val_struct,
        training_strategy="DoRA",
        prompt_format="structOnly",
        model_name=MODEL_ID,
        output_csv=f"{ARTIFACTS_DIR}/experiment_summary.csv"
    )
    
    # Unload adapter
    model_struct.unload()
except Exception as e:
    print(f"Could not load or evaluate Structural-Only SFT model: {e}")


Loading adapter from ..//output/models/granite-4.1-8b_structOnly_DoRA...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Evaluating DoRA - structOnly:   0%|          | 0/2039 [00:00<?, ?it/s]

/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Evaluating DoRA - structOnly:   1%|          | 20/2039 [00:52<1:27:41,  2.61s/it]

...

Evaluating DoRA - structOnly:  97%|█████████▋| 1980/2039 [1:23:18<02:24,  2.44s/it]

Evaluating DoRA - structOnly:  98%|█████████▊| 2000/2039 [1:24:09<01:36,  2.48s/it]

Evaluating DoRA - structOnly:  99%|█████████▉| 2020/2039 [1:24:58<00:46,  2.46s/it]

Evaluating DoRA - structOnly: 100%|██████████| 2039/2039 [1:25:44<00:00,  2.52s/it]

\n--- Evaluation Results ---
Training Strategy: DoRA
Prompt Format: structOnly
Model: ibm-granite/granite-4.1-8b
Accuracy: 0.7798
Format Error Rate: 0.2217
Semantic Confusion: 0.3185
Option Bias (A): 0.3296
Latency: 5144.50 seconds
Detailed predictions saved to: /data220_2/emmy/mlbio/hw4/output/validation/validation_granite-4.1-8b_structOnly_DoRA.csv
